# 🚀 Elastic 9.x Vector Search: 사전 데이터 적재 (Setup)
본 노트북은 데모 시연을 위한 한국어 오픈 데이터셋(KLUE) 다운로드, 전처리 및 Elastic 클러스터 고속 색인을 수행합니다.
안전한 시연을 위해 SSL 경고를 숨기고 사용자 인증을 수행합니다.

In [ ]:
import os
import time
import random
import getpass
import warnings
import urllib3
from collections import deque
from datasets import load_dataset
from elasticsearch import Elasticsearch, helpers
from elasticsearch.exceptions import ElasticsearchWarning

# 경고 메시지 숨김 (데모용)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.simplefilter('ignore', ElasticsearchWarning)

print("🔌 Elasticsearch 클러스터 연결 정보를 입력하세요.")
es_url = os.environ.get("ES_URL") or input("ES URL [기본값: https://localhost:9200]: ") or "https://localhost:9200"
es_user = os.environ.get("ES_USER") or input("Username [기본값: elastic]: ") or "elastic"
es_pass = os.environ.get("ES_PASSWORD") or getpass.getpass("Password: ")

es = Elasticsearch(es_url, basic_auth=(es_user, es_pass), verify_certs=False)
if es.ping():
    print("✅ 클러스터 연결 성공!")
else:
    print("❌ 연결 실패. 접속 정보를 확인하세요.")

INDEX_REAL_LEGACY = "demo-real-legacy-8x"
INDEX_REAL_MODERN = "demo-real-modern-9x"
VECTOR_DIMS = 1024

## 1. Elastic Inference Service (EIS) 세팅
외부 API 호출 없이 클러스터 내부에서 안전하게 동작하는 Jina v5 (임베딩 및 리랭커) 모델을 활성화합니다.

In [ ]:
import time
from elasticsearch.exceptions import NotFoundError

def deploy_model_if_not_exists(inference_id, task_type, model_config):
    try:
        # 1. 모델이 이미 존재하는지 확인
        es.inference.get(inference_id=inference_id)
        print(f"✅ 모델 '{inference_id}'가 이미 존재합니다. 배포를 건너뜁니다.")
    except NotFoundError:
        # 2. 존재하지 않는 경우에만 배포 시도
        print(f"🚀 모델 '{inference_id}'이 없습니다. 배포를 시작합니다...")
        try:
            es.inference.put(
                task_type=task_type, 
                inference_id=inference_id, 
                body=model_config,
                request_timeout=60  # 타임아웃 에러 방지를 위해 대기 시간 연장
            )
            print(f"⏳ 모델 '{inference_id}' 배포 명령 완료. 활성화를 기다리는 중...")
            
            # 3. 모델이 실제로 사용 가능한 상태가 될 때까지 짧은 대기 (데모 안정성)
            time.sleep(5) 
        except Exception as e:
            print(f"❌ 모델 배포 중 에러 발생: {e}")

# Jina v5 임베딩 모델 설정
embedding_config = {
    "service": "elastic", 
    "service_settings": {"model_id": "jina-embeddings-v5-text-small"}
}

# Jina 리랭커 모델 설정
reranker_config = {
    "service": "elastic", 
    "service_settings": {"model_id": "jina-reranker-v3"}
}

# 함수 실행
deploy_model_if_not_exists("my-jina-embedding", "text_embedding", embedding_config)
deploy_model_if_not_exists("my-jina-reranker", "rerank", reranker_config)

print("\n✨ 모든 Inference 모델 확인 및 배포 절차가 완료되었습니다.")

## 2. A/B 테스트용 인덱스 아키텍처 구성
* **Legacy (8.x)**: 무거운 `float32` 타입과 `Ingest Pipeline`을 사용한 과거의 복잡한 방식
* **Modern (9.x)**: `semantic_text` 필드 하나로 `bfloat16` 저장소 압축과 `BBQ` 메모리 양자화를 자동화하는 모던 방식

In [ ]:
print("🏗️ 완벽하게 통제된 A/B 테스트용 인덱스 아키텍처 재구성 중...")
es.indices.delete(index=[INDEX_REAL_LEGACY, INDEX_REAL_MODERN], ignore_unavailable=True)

# 1. Legacy Ingest Pipeline
es.ingest.put_pipeline(id="legacy-inference-pipeline", body={
    "processors": [{
        "inference": {
            "model_id": "my-jina-embedding",
            "input_output": [{"input_field": "title", "output_field": "title_vector"}]
        }
    }]
})

# 2. [Legacy] 인덱스: 양자화(BBQ/int8)를 완전히 배제한 순수 float + HNSW
es.indices.create(index=INDEX_REAL_LEGACY, body={
    "mappings": {
        "properties": {
            "title": {"type": "text"},
            "title_vector": {
                "type": "dense_vector", 
                "dims": VECTOR_DIMS, 
                "element_type": "float", 
                "similarity": "cosine", 
                "index": True,
                "index_options": {
                    "type": "hnsw" # 🔥 엔진의 자동 양자화를 막고 순수 HNSW 강제
                }
            },
            "category": {"type": "keyword"},
            "access_level": {"type": "keyword"}
        }
    }
})

# 3. [Modern] 인덱스: 명시적 BBQ 적용 및 청킹 오버헤드 제거
es.indices.create(index=INDEX_REAL_MODERN, body={
    "mappings": {
        "properties": {
            "content_semantic": {
                "type": "semantic_text", 
                "inference_id": "my-jina-embedding",
                "index_options": {
                    "dense_vector": {
                        "type": "bbq_hnsw" # 🔥 최신 1-bit BBQ 양자화 명시적 선언
                    }
                },
                "chunking_settings": {
                    "strategy": "none" # 🔥 올바른 파라미터명 수정 (type -> strategy)
                }
            },
            "title": {"type": "text", "copy_to": "content_semantic"},
            "category": {"type": "keyword"},
            "access_level": {"type": "keyword"}
        }
    }
})
print("✅ 아키텍처 구성 완료 (통제된 A/B 테스트 환경)")

## 3. 한국어 오픈 데이터(KLUE) 고속 색인 및 희소 필터(1%) 태깅
ACORN-1 알고리즘의 벤치마크를 위해 전체 문서 중 딱 **1%**에만 `access_level: premium` 태그를 부여하여 극한의 필터링 환경을 조성합니다.

In [ ]:
from datasets import load_dataset, utils
import random
from elasticsearch import helpers

utils.logging.set_verbosity_error()
print("🚀 KLUE 한국어 뉴스 데이터 다운로드 및 색인 중...")

dataset = load_dataset("klue", "ynat", split="train")
label_map = {0: "IT/과학", 1: "경제", 2: "사회", 3: "생활문화", 4: "세계", 5: "스포츠", 6: "정치"}

# 1. 빈 텍스트(Dirty Data)를 사전에 걸러내는 로직 추가
def generate_real_data(index_name, use_pipeline=False):
    for i, row in enumerate(dataset):
        title = row.get("title", "")
        if not title or len(title.strip()) == 0:
            continue # 제목이 비어있는 불량 데이터는 건너뜁니다.
            
        access_level = "premium" if random.random() < 0.01 else "public"
        action = {
            "_index": index_name, "_id": str(i),
            "_source": {"title": title, "category": label_map.get(row["label"], "기타"), "access_level": access_level}
        }
        if use_pipeline: action["pipeline"] = "legacy-inference-pipeline"
        yield action

# 2. 색인 함수 (개별 문서 실패 시 멈추지 않고 패스하는 Production 설정)
def run_indexing(index_name, pipeline=False):
    count = 0
    fail_count = 0
    # 🔥 raise_on_error=False 옵션으로 에러가 나도 파이프라인이 멈추지 않게 보호합니다.
    for success, info in helpers.parallel_bulk(
        es, 
        generate_real_data(index_name, pipeline), 
        thread_count=2, 
        chunk_size=50, 
        request_timeout=120,
        raise_on_error=False,
        raise_on_exception=False
    ):
        count += 1
        if not success:
            fail_count += 1
        if count % 5000 == 0: 
            print(f"   -> {count}건 처리 완료... (건너뛴 불량 데이터: {fail_count}건)")
            
    print(f"✅ 적재 완료! (총 {count}건 처리 / 불량 데이터 {fail_count}건 패스)")

print("⏳ [Legacy] 인덱스 적재 중... (ML 임베딩으로 인해 시간이 다소 소요됩니다)")
run_indexing(INDEX_REAL_LEGACY, pipeline=True)

print("⏳ [Modern] 인덱스 적재 중...")
run_indexing(INDEX_REAL_MODERN, pipeline=False)

print("⚙️ 세그먼트 최적화 수행 중...")
es.indices.forcemerge(index=[INDEX_REAL_LEGACY, INDEX_REAL_MODERN], max_num_segments=1, request_timeout=300)
es.indices.refresh(index=[INDEX_REAL_LEGACY, INDEX_REAL_MODERN])
print("🎉 모든 데이터 준비가 완벽하게 완료되었습니다!")